# 04_handover_data_contract_export
Generate a metadata-backed handover/data contract package and export YAML/JSON.


In [ ]:
from fabricops_kit.config import setup_fabricops_notebook, get_path
from fabricops_kit.fabric_input_output import lakehouse_table_read
from fabricops_kit import generate_handover_contract, export_handover_contract


In [ ]:
ENV_NAME='dev'
DATASET_NAME='YOUR_DATASET'
TARGET_TABLE='YOUR_TARGET_TABLE'
EXPORT_FORMAT='yaml'  # yaml or json
EXPORT_PATH='/lakehouse/default/Files/handover_contract.yaml'
ctx = setup_fabricops_notebook()
spark = ctx['spark']
metadata_path = get_path(ENV_NAME, 'metadata')


## Load approved metadata and evidence
This step reads contracts, columns, rules, quality results, and lineage records.


In [ ]:
contracts = [r.asDict(recursive=True) for r in lakehouse_table_read(metadata_path, 'contracts').collect()]
contract_columns = [r.asDict(recursive=True) for r in lakehouse_table_read(metadata_path, 'contract_columns').collect()]
contract_rules = [r.asDict(recursive=True) for r in lakehouse_table_read(metadata_path, 'contract_rules').collect()]
quality_results = [r.asDict(recursive=True) for r in lakehouse_table_read(metadata_path, 'quality_results').collect()]
lineage_records = [r.asDict(recursive=True) for r in lakehouse_table_read(metadata_path, 'lineage_records').collect()]


## Generate and export handover package
AI may be used only for narrative text. Approved metadata stays read-only.


In [ ]:
handover = generate_handover_contract(
    contracts=contracts,
    contract_columns=contract_columns,
    contract_rules=contract_rules,
    quality_results=quality_results,
    lineage_records=lineage_records,
    dataset_name=DATASET_NAME,
    table_name=TARGET_TABLE,
    ai_narrative='',
    include_prompt=True,
)
display(handover['summary'])
written_path = export_handover_contract(handover, EXPORT_PATH, format=EXPORT_FORMAT)
print('Exported handover package:', written_path)
